# 14. Fama-French 三因子模型

## 学习目标

- 理解 Fama-French 三因子模型的理论基础
- 掌握 SMB（规模因子）和 HML（价值因子）的构建方法
- 学会用 Python 实现因子构建和回归分析
- 能够解读回归结果（t值、R²、因子显著性）

## 环境依赖

```bash
pip install numpy pandas matplotlib scipy statsmodels akshare
```

## 一、理论基础

### 1.1 什么是 Fama-French 三因子模型？

Fama-French 三因子模型是由 Eugene Fama 和 Kenneth French 于 1993 年提出的资产定价模型，它是对 CAPM 模型的扩展。

模型公式：

$$R_i - R_f = \alpha_i + \beta_i(R_m - R_f) + s_i \cdot SMB + h_i \cdot HML + \epsilon_i$$

其中：
- $R_i$：资产 i 的收益率
- $R_f$：无风险利率
- $R_m$：市场组合收益率
- $SMB$：Small Minus Big，规模因子（小市值 - 大市值）
- $HML$：High Minus Low，价值因子（高账面市值比 - 低账面市值比）
- $\beta_i$：市场风险暴露
- $s_i$：规模因子暴露
- $h_i$：价值因子暴露

### 1.2 三个因子的含义

| 因子 | 含义 | 经济解释 |
|------|------|----------|
| $R_m - R_f$ | 市场超额收益 | 承担市场风险的补偿 |
| $SMB$ | 规模因子 | 小公司相对大公司的超额收益 |
| $HML$ | 价值因子 | 价值股相对成长股的超额收益 |

### 1.3 为什么需要三因子模型？

CAPM 模型只考虑市场风险（单因子），但实证研究发现：
- **小市值效应**：小公司股票长期收益高于大公司
- **价值效应**：高 B/M 比（价值股）收益高于低 B/M 比（成长股）

三因子模型捕获了这些异象，能更好地解释股票收益的横截面差异。

## 二、因子构建方法

### 2.1 SMB（Small Minus Big）构建步骤

1. 每年 6 月底，按市值中位数将所有股票分为 **Small (S)** 和 **Big (B)** 两组
2. 计算每组的市值加权收益率
3. $SMB = R_S - R_B$

### 2.2 HML（High Minus Low）构建步骤

1. 每年 6 月底，按 B/M 比的 30% 和 70% 分位数将股票分为三组：
   - **High (H)**：B/M > 70% 分位数（价值股）
   - **Middle (M)**：30% < B/M < 70%
   - **Low (L)**：B/M < 30% 分位数（成长股）
2. $HML = R_H - R_L$

### 2.3 市值加权收益率

$$R_{portfolio} = \sum_{i} w_i \cdot R_i$$

其中 $w_i = \frac{MarketCap_i}{\sum MarketCap}$

## 三、Python 实现

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import statsmodels.api as sm
import warnings
warnings.filterwarnings('ignore')

# 设置中文显示
plt.rcParams['font.sans-serif'] = ['Arial Unicode MS', 'SimHei']
plt.rcParams['axes.unicode_minus'] = False

print('环境准备完成！')

### 3.1 获取模拟数据

由于真实财务数据获取较复杂，我们先用模拟数据演示因子构建和回归的完整流程。

In [ ]:
def generate_mock_data(n_stocks=100, n_periods=60):
    """
    生成模拟股票数据
    n_stocks: 股票数量
    n_periods: 时间段数量（月）
    """
    np.random.seed(42)
    
    dates = pd.date_range('2019-01-01', periods=n_periods, freq='ME')
    stock_ids = [f'Stock_{i:03d}' for i in range(n_stocks)]
    
    # 生成市值（对数正态分布，模拟真实市值分布）
    market_caps = np.exp(np.random.normal(10, 2, n_stocks))  # 市值
    
    # 生成 B/M 比（账面市值比）
    bm_ratios = np.random.lognormal(-1, 0.8, n_stocks)  # B/M 比
    bm_ratios = np.clip(bm_ratios, 0.05, 5)  # 限制范围
    
    # 生成收益率（与市值和 B/M 相关）
    # 小市值有溢价，高 B/M 有溢价
    market_returns = np.random.normal(0.008, 0.05, n_periods)  # 市场收益
    
    returns_data = []
    for i in range(n_stocks):
        # 小市值股票 beta 更高
        size_effect = -0.3 * (np.log(market_caps[i]) - 10) / 2
        # 高 B/M 股票有溢价
        value_effect = 0.2 * (np.log(bm_ratios[i]) + 1) / 2
        
        stock_returns = (
            0.001  # alpha
            + 1.0 * market_returns  # 市场 beta
            + size_effect * 0.02  # 规模效应
            + value_effect * 0.02  # 价值效应
            + np.random.normal(0, 0.03, n_periods)  # 个股噪声
        )
        
        for t in range(n_periods):
            returns_data.append({
                'date': dates[t],
                'stock_id': stock_ids[i],
                'return': stock_returns[t],
                'market_cap': market_caps[i],
                'bm_ratio': bm_ratios[i]
            })
    
    df = pd.DataFrame(returns_data)
    df['date'] = pd.to_datetime(df['date'])
    return df, market_returns, dates

# 生成数据
df, market_returns, dates = generate_mock_data()
print(f"数据维度: {df.shape}")
print(f"时间范围: {df['date'].min()} 到 {df['date'].max()}")
print(f"股票数量: {df['stock_id'].nunique()}")
print(f"\n前5行数据:")
df.head()

### 3.2 构建 SMB 因子

In [ ]:
def build_smb_factor(df):
    """
    构建 SMB (Small Minus Big) 因子
    按市值中位数分为小市值和大市值两组，计算收益率之差
    """
    smb_results = []
    
    for date, group in df.groupby('date'):
        # 计算市值中位数
        median_cap = group['market_cap'].median()
        
        # 分组
        small_stocks = group[group['market_cap'] <= median_cap]
        big_stocks = group[group['market_cap'] > median_cap]
        
        # 市值加权收益率
        def weighted_return(sub_df):
            weights = sub_df['market_cap'] / sub_df['market_cap'].sum()
            return (sub_df['return'] * weights).sum()
        
        r_small = weighted_return(small_stocks)
        r_big = weighted_return(big_stocks)
        
        smb_results.append({
            'date': date,
            'SMB': r_small - r_big,
            'R_small': r_small,
            'R_big': r_big
        })
    
    return pd.DataFrame(smb_results)

smb_df = build_smb_factor(df)
print(f"SMB 因子统计:")
print(smb_df['SMB'].describe())
print(f"\nSMB 均值: {smb_df['SMB'].mean():.4f} ({smb_df['SMB'].mean()*12*100:.2f}% 年化)")

### 3.3 构建 HML 因子

In [ ]:
def build_hml_factor(df):
    """
    构建 HML (High Minus Low) 因子
    按 B/M 比的 30% 和 70% 分位数分为三组
    """
    hml_results = []
    
    for date, group in df.groupby('date'):
        # 计算 B/M 分位数
        q30 = group['bm_ratio'].quantile(0.3)
        q70 = group['bm_ratio'].quantile(0.7)
        
        # 分组
        high_bm = group[group['bm_ratio'] >= q70]   # 价值股
        low_bm = group[group['bm_ratio'] <= q30]    # 成长股
        
        # 市值加权收益率
        def weighted_return(sub_df):
            weights = sub_df['market_cap'] / sub_df['market_cap'].sum()
            return (sub_df['return'] * weights).sum()
        
        r_high = weighted_return(high_bm)
        r_low = weighted_return(low_bm)
        
        hml_results.append({
            'date': date,
            'HML': r_high - r_low,
            'R_high_bm': r_high,
            'R_low_bm': r_low
        })
    
    return pd.DataFrame(hml_results)

hml_df = build_hml_factor(df)
print(f"HML 因子统计:")
print(hml_df['HML'].describe())
print(f"\nHML 均值: {hml_df['HML'].mean():.4f} ({hml_df['HML'].mean()*12*100:.2f}% 年化)")

### 3.4 构建市场超额收益

In [ ]:
# 假设无风险利率为 3% 年化
rf_annual = 0.03
rf_monthly = rf_annual / 12

# 构建市场超额收益
market_df = pd.DataFrame({
    'date': dates,
    'Rm': market_returns,
    'Rf': rf_monthly,
    'Rm_Rf': market_returns - rf_monthly  # 市场超额收益
})

print(f"市场超额收益统计:")
print(market_df['Rm_Rf'].describe())

### 3.5 合并因子数据

In [ ]:
# 合并所有因子
factors_df = market_df[['date', 'Rm_Rf']].merge(smb_df[['date', 'SMB']], on='date')
factors_df = factors_df.merge(hml_df[['date', 'HML']], on='date')

print("因子数据:")
print(factors_df.head())
print(f"\n因子相关系数矩阵:")
print(factors_df[['Rm_Rf', 'SMB', 'HML']].corr().round(3))

## 四、三因子回归分析

### 4.1 选择一个投资组合进行回归

In [ ]:
# 构建一个投资组合（等权重持有所有股票）
def build_portfolio_returns(df):
    """构建等权重投资组合收益率"""
    portfolio_returns = df.groupby('date')['return'].mean().reset_index()
    portfolio_returns.columns = ['date', 'R_portfolio']
    return portfolio_returns

portfolio_df = build_portfolio_returns(df)

# 合并因子数据
regression_data = portfolio_df.merge(factors_df, on='date')
regression_data['R_p_Rf'] = regression_data['R_portfolio'] - rf_monthly  # 组合超额收益

print("回归数据:")
print(regression_data.head())

### 4.2 执行 OLS 回归

In [ ]:
# 三因子回归: R_p - Rf = alpha + beta*(Rm-Rf) + s*SMB + h*HML + epsilon
X = regression_data[['Rm_Rf', 'SMB', 'HML']]
X = sm.add_constant(X)  # 添加截距项
y = regression_data['R_p_Rf']

# OLS 回归
model = sm.OLS(y, X).fit()

print("=" * 60)
print("Fama-French 三因子回归结果")
print("=" * 60)
print(model.summary())

### 4.3 解读回归结果

In [ ]:
print("\n" + "=" * 60)
print("回归结果解读")
print("=" * 60)

# 提取关键指标
alpha = model.params['const']
beta = model.params['Rm_Rf']
s = model.params['SMB']
h = model.params['HML']

t_alpha = model.tvalues['const']
t_beta = model.tvalues['Rm_Rf']
t_s = model.tvalues['SMB']
t_h = model.tvalues['HML']

p_alpha = model.pvalues['const']
p_beta = model.pvalues['Rm_Rf']
p_s = model.pvalues['SMB']
p_h = model.pvalues['HML']

r_squared = model.rsquared
adj_r_squared = model.rsquared_adj

print(f"\n1. Alpha (截距):")
print(f"   系数: {alpha:.6f} (年化: {alpha*12*100:.2f}%)")
print(f"   t值: {t_alpha:.3f}, p值: {p_alpha:.4f}")
print(f"   {'✓ 显著' if p_alpha < 0.05 else '✗ 不显著'} (α=0.05)")

print(f"\n2. 市场因子 (Rm-Rf):")
print(f"   Beta: {beta:.4f}")
print(f"   t值: {t_beta:.3f}, p值: {p_beta:.4f}")
print(f"   {'✓ 显著' if p_beta < 0.05 else '✗ 不显著'} (α=0.05)")

print(f"\n3. 规模因子 (SMB):")
print(f"   系数 s: {s:.4f}")
print(f"   t值: {t_s:.3f}, p值: {p_s:.4f}")
print(f"   {'✓ 显著' if p_s < 0.05 else '✗ 不显著'} (α=0.05)")
if s > 0:
    print(f"   → 组合偏向小市值股票")
else:
    print(f"   → 组合偏向大市值股票")

print(f"\n4. 价值因子 (HML):")
print(f"   系数 h: {h:.4f}")
print(f"   t值: {t_h:.3f}, p值: {p_h:.4f}")
print(f"   {'✓ 显著' if p_h < 0.05 else '✗ 不显著'} (α=0.05)")
if h > 0:
    print(f"   → 组合偏向价值股（高 B/M）")
else:
    print(f"   → 组合偏向成长股（低 B/M）")

print(f"\n5. 模型拟合度:")
print(f"   R²: {r_squared:.4f} ({r_squared*100:.1f}%)")
print(f"   调整R²: {adj_r_squared:.4f} ({adj_r_squared*100:.1f}%)")
print(f"   → 三因子模型解释了组合收益 {r_squared*100:.1f}% 的变异")

## 五、可视化分析

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. 因子累积收益
ax1 = axes[0, 0]
cumulative_smb = (1 + smb_df['SMB']).cumprod()
cumulative_hml = (1 + hml_df['HML']).cumprod()
cumulative_rm = (1 + market_df['Rm_Rf']).cumprod()

ax1.plot(smb_df['date'], cumulative_smb, label='SMB', linewidth=2)
ax1.plot(hml_df['date'], cumulative_hml, label='HML', linewidth=2)
ax1.plot(market_df['date'], cumulative_rm, label='Rm-Rf', linewidth=2)
ax1.set_title('因子累积收益 (2019-2023)', fontsize=12)
ax1.set_xlabel('日期')
ax1.set_ylabel('累积收益')
ax1.legend()
ax1.grid(True, alpha=0.3)

# 2. 因子相关性热力图
ax2 = axes[0, 1]
corr_matrix = factors_df[['Rm_Rf', 'SMB', 'HML']].corr()
im = ax2.imshow(corr_matrix, cmap='RdYlBu_r', vmin=-1, vmax=1)
ax2.set_xticks(range(3))
ax2.set_yticks(range(3))
ax2.set_xticklabels(['Rm-Rf', 'SMB', 'HML'])
ax2.set_yticklabels(['Rm-Rf', 'SMB', 'HML'])
ax2.set_title('因子相关系数矩阵', fontsize=12)
for i in range(3):
    for j in range(3):
        ax2.text(j, i, f'{corr_matrix.iloc[i, j]:.2f}',
                ha='center', va='center', fontsize=12)
plt.colorbar(im, ax=ax2)

# 3. 实际收益 vs 预测收益
ax3 = axes[1, 0]
predicted = model.predict(X)
ax3.scatter(predicted, y, alpha=0.5, s=30)
ax3.plot([-0.1, 0.1], [-0.1, 0.1], 'r--', linewidth=2, label='完美预测线')
ax3.set_xlabel('预测超额收益')
ax3.set_ylabel('实际超额收益')
ax3.set_title(f'实际 vs 预测 (R²={r_squared:.3f})', fontsize=12)
ax3.legend()
ax3.grid(True, alpha=0.3)

# 4. 回归残差
ax4 = axes[1, 1]
residuals = model.resid
ax4.bar(range(len(residuals)), residuals, alpha=0.6, color='steelblue')
ax4.axhline(y=0, color='r', linestyle='--', linewidth=1)
ax4.set_xlabel('观测序号')
ax4.set_ylabel('残差')
ax4.set_title('回归残差分布', fontsize=12)
ax4.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('fama_french_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print("图表已保存: fama_french_analysis.png")

## 六、分组检验：规模和价值的十分位组合

In [ ]:
def decile_analysis(df, factor_col, factor_name):
    """
    按因子分十分位，检验收益差异
    """
    # 计算每个股票的平均因子值
    stock_factor = df.groupby('stock_id')[factor_col].mean()
    
    # 分十分位
    stock_factor_decile = pd.qcut(stock_factor, 10, labels=False) + 1
    
    # 计算每个十分位的平均收益
    stock_returns = df.groupby('stock_id')['return'].mean()
    
    decile_returns = []
    for d in range(1, 11):
        stocks_in_decile = stock_factor_decile[stock_factor_decile == d].index
        avg_return = stock_returns[stocks_in_decile].mean()
        decile_returns.append({
            'decile': d,
            'avg_return': avg_return,
            'n_stocks': len(stocks_in_decile)
        })
    
    return pd.DataFrame(decile_returns)

# 按市值分十分位
size_decile = decile_analysis(df, 'market_cap', '市值')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 市值十分位收益
ax1 = axes[0]
colors = ['#d32f2f' if x < 5 else '#1976d2' for x in size_decile['decile']]
ax1.bar(size_decile['decile'], size_decile['avg_return'] * 100, color=colors, alpha=0.8)
ax1.set_xlabel('市值十分位 (1=最小, 10=最大)')
ax1.set_ylabel('月均收益率 (%)')
ax1.set_title('按市值分组的平均收益', fontsize=12)
ax1.grid(True, alpha=0.3, axis='y')

# B/M 十分位收益
bm_decile = decile_analysis(df, 'bm_ratio', 'B/M')
ax2 = axes[1]
colors = ['#d32f2f' if x < 5 else '#1976d2' for x in bm_decile['decile']]
ax2.bar(bm_decile['decile'], bm_decile['avg_return'] * 100, color=colors, alpha=0.8)
ax2.set_xlabel('B/M 十分位 (1=最低, 10=最高)')
ax2.set_ylabel('月均收益率 (%)')
ax2.set_title('按 B/M 分组的平均收益', fontsize=12)
ax2.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('decile_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"小市值组合月均收益: {size_decile.iloc[0]['avg_return']*100:.3f}%")
print(f"大市值组合月均收益: {size_decile.iloc[-1]['avg_return']*100:.3f}%")
print(f"SMB 溢价: {(size_decile.iloc[0]['avg_return'] - size_decile.iloc[-1]['avg_return'])*100:.3f}%")

## 七、小结

### 核心要点

1. **Fama-French 三因子模型**：$R_i - R_f = \alpha + \beta(R_m - R_f) + s \cdot SMB + h \cdot HML + \epsilon$
2. **SMB 因子**：按市值分组，小市值组合收益 - 大市值组合收益
3. **HML 因子**：按 B/M 比分组，高 B/M（价值股）收益 - 低 B/M（成长股）收益
4. **回归解读**：
   - **t值 > 2** 或 **p值 < 0.05** → 因子显著
   - **R²** → 模型解释力
   - **Alpha** → 超额收益（应接近 0）

### 验收标准 Checklist

- [x] 能手工构建 SMB 和 HML 因子
- [x] 理解做多/做空的构建逻辑（Long-Short）
- [x] 能读取回归结果（t值、R²、p值）
- [x] 理解因子暴露的经济含义